In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder \
    .appName("Phase 4A - Bucketing") \
    .getOrCreate()

In [3]:
from google.colab import files

uploaded = files.upload()

Saving sales.csv to sales.csv


In [4]:
sales_df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

In [5]:
sales_df.show(5)

+-------+-----------+----------+----------+--------+------------+
|sale_id|customer_id|product_id| sale_date|quantity|total_amount|
+-------+-----------+----------+----------+--------+------------+
|      1|          1|         1|15-01-2024|       2|       39.98|
|      2|          1|         3|20-01-2024|       1|       29.99|
|      3|          2|         2|16-01-2024|       1|        25.0|
|      4|          2|         4|22-01-2024|       3|       89.97|
|      5|          3|         5|17-01-2024|       2|       49.98|
+-------+-----------+----------+----------+--------+------------+
only showing top 5 rows


In [6]:
sales_df.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)



##Clean Null Values

In [7]:
sales_clean = sales_df.dropna()

##Calculate Total Spend Per Customer

In [8]:
customer_spend = sales_clean.groupBy("customer_id") \
    .agg(
        sum("total_amount").alias("total_spend")
    )

customer_spend.show()

+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|         31|       50.0|
|         34|       92.0|
|         28|       45.0|
|         26|      66.75|
|         27|      79.96|
|         44|       55.5|
|         12|      67.47|
|         22|     119.96|
|          1|      69.97|
|         13|       34.0|
|          6|       40.0|
|         16|       60.0|
|          3|      49.98|
|         20|      89.97|
|         40|       40.0|
|          5|      66.75|
|         19|      29.99|
|         41|      67.47|
|         15|       92.0|
|         43|       40.0|
+-----------+-----------+
only showing top 20 rows


##Practice Task 1

####SQL

In [9]:
customer_spend.createOrReplaceTempView("customer_spend")

In [10]:
spark.sql("""
SELECT
customer_id,
total_spend,
CASE
WHEN total_spend > 100 THEN 'Gold'
WHEN total_spend >= 50 THEN 'Silver'
ELSE 'Bronze'
END AS segment
FROM customer_spend
""").show()

+-----------+-----------+-------+
|customer_id|total_spend|segment|
+-----------+-----------+-------+
|         31|       50.0| Silver|
|         34|       92.0| Silver|
|         28|       45.0| Bronze|
|         26|      66.75| Silver|
|         27|      79.96| Silver|
|         44|       55.5| Silver|
|         12|      67.47| Silver|
|         22|     119.96|   Gold|
|          1|      69.97| Silver|
|         13|       34.0| Bronze|
|          6|       40.0| Bronze|
|         16|       60.0| Silver|
|          3|      49.98| Bronze|
|         20|      89.97| Silver|
|         40|       40.0| Bronze|
|          5|      66.75| Silver|
|         19|      29.99| Bronze|
|         41|      67.47| Silver|
|         15|       92.0| Silver|
|         43|       40.0| Bronze|
+-----------+-----------+-------+
only showing top 20 rows


###PySpark

In [11]:
segmented_df = customer_spend.withColumn(
    "segment",
    when(col("total_spend") > 100, "Gold")
    .when(col("total_spend") >= 50, "Silver")
    .otherwise("Bronze")
)

segmented_df.show()

+-----------+-----------+-------+
|customer_id|total_spend|segment|
+-----------+-----------+-------+
|         31|       50.0| Silver|
|         34|       92.0| Silver|
|         28|       45.0| Bronze|
|         26|      66.75| Silver|
|         27|      79.96| Silver|
|         44|       55.5| Silver|
|         12|      67.47| Silver|
|         22|     119.96|   Gold|
|          1|      69.97| Silver|
|         13|       34.0| Bronze|
|          6|       40.0| Bronze|
|         16|       60.0| Silver|
|          3|      49.98| Bronze|
|         20|      89.97| Silver|
|         40|       40.0| Bronze|
|          5|      66.75| Silver|
|         19|      29.99| Bronze|
|         41|      67.47| Silver|
|         15|       92.0| Silver|
|         43|       40.0| Bronze|
+-----------+-----------+-------+
only showing top 20 rows


#####Practice Task 2 - Count Customers in Each Segment

SQL

In [12]:
segmented_df.createOrReplaceTempView("segmented")

In [13]:
spark.sql("""
SELECT
segment,
COUNT(*) AS customer_count
FROM segmented
GROUP BY segment
""").show()

+-------+--------------+
|segment|customer_count|
+-------+--------------+
| Silver|            19|
|   Gold|             5|
| Bronze|            22|
+-------+--------------+



##PySpark

In [14]:
segmented_df.groupBy("segment") \
    .count() \
    .show()

+-------+-----+
|segment|count|
+-------+-----+
| Silver|   19|
|   Gold|    5|
| Bronze|   22|
+-------+-----+



Practice Task 3
Quantile-Based Segmentation

Find 33% and 66% spending values.

In [15]:
quantiles = customer_spend.approxQuantile(
    "total_spend",
    [0.33, 0.66],
    0
)

print(quantiles)

[40.0, 67.47]


In [16]:
quantile_df = customer_spend.withColumn(
    "segment",
    when(col("total_spend") <= quantiles[0], "Bronze")
    .when(col("total_spend") <= quantiles[1], "Silver")
    .otherwise("Gold")
)

quantile_df.show()

+-----------+-----------+-------+
|customer_id|total_spend|segment|
+-----------+-----------+-------+
|         31|       50.0| Silver|
|         34|       92.0|   Gold|
|         28|       45.0| Silver|
|         26|      66.75| Silver|
|         27|      79.96|   Gold|
|         44|       55.5| Silver|
|         12|      67.47| Silver|
|         22|     119.96|   Gold|
|          1|      69.97|   Gold|
|         13|       34.0| Bronze|
|          6|       40.0| Bronze|
|         16|       60.0| Silver|
|          3|      49.98| Silver|
|         20|      89.97|   Gold|
|         40|       40.0| Bronze|
|          5|      66.75| Silver|
|         19|      29.99| Bronze|
|         41|      67.47| Silver|
|         15|       92.0|   Gold|
|         43|       40.0| Bronze|
+-----------+-----------+-------+
only showing top 20 rows


Practice Task 4:
Bucketizer (MLlib)


In [17]:
from pyspark.ml.feature import Bucketizer

splits = [-float("inf"),50,100,float("inf")]

bucketizer = Bucketizer(
    splits=splits,
    inputCol="total_spend",
    outputCol="bucket"
)

bucket_df = bucketizer.transform(customer_spend)

bucket_df.show()

+-----------+-----------+------+
|customer_id|total_spend|bucket|
+-----------+-----------+------+
|         31|       50.0|   1.0|
|         34|       92.0|   1.0|
|         28|       45.0|   0.0|
|         26|      66.75|   1.0|
|         27|      79.96|   1.0|
|         44|       55.5|   1.0|
|         12|      67.47|   1.0|
|         22|     119.96|   2.0|
|          1|      69.97|   1.0|
|         13|       34.0|   0.0|
|          6|       40.0|   0.0|
|         16|       60.0|   1.0|
|          3|      49.98|   0.0|
|         20|      89.97|   1.0|
|         40|       40.0|   0.0|
|          5|      66.75|   1.0|
|         19|      29.99|   0.0|
|         41|      67.47|   1.0|
|         15|       92.0|   1.0|
|         43|       40.0|   0.0|
+-----------+-----------+------+
only showing top 20 rows


Practice Task 5:
Window-Based Ranking
######SQL

In [18]:
customer_spend.createOrReplaceTempView("customer_spend")
spark.sql("""
SELECT
customer_id,
total_spend,
PERCENT_RANK() OVER(ORDER BY total_spend) AS rank_percent
FROM customer_spend
""").show()

+-----------+-----------+--------------------+
|customer_id|total_spend|        rank_percent|
+-----------+-----------+--------------------+
|         46|      12.99|                 0.0|
|         14|       15.0|0.022222222222222223|
|         23|       15.5|0.044444444444444446|
|         36|      18.75| 0.06666666666666667|
|         38|      19.99| 0.08888888888888889|
|          8|       20.0|  0.1111111111111111|
|         42|       22.5| 0.13333333333333333|
|         11|       25.0| 0.15555555555555556|
|         32|       27.5| 0.17777777777777778|
|         19|      29.99|                 0.2|
|         13|       34.0|  0.2222222222222222|
|         33|       34.0|  0.2222222222222222|
|         29|      35.98| 0.26666666666666666|
|         37|      39.98| 0.28888888888888886|
|          6|       40.0|  0.3111111111111111|
|         40|       40.0|  0.3111111111111111|
|         43|       40.0|  0.3111111111111111|
|         17|       40.0|  0.3111111111111111|
|         25|

PySpark

In [19]:
window = Window.orderBy("total_spend")

rank_df = customer_spend.withColumn(
    "rank_percent",
    percent_rank().over(window)
)

rank_df.show()

+-----------+-----------+--------------------+
|customer_id|total_spend|        rank_percent|
+-----------+-----------+--------------------+
|         46|      12.99|                 0.0|
|         14|       15.0|0.022222222222222223|
|         23|       15.5|0.044444444444444446|
|         36|      18.75| 0.06666666666666667|
|         38|      19.99| 0.08888888888888889|
|          8|       20.0|  0.1111111111111111|
|         42|       22.5| 0.13333333333333333|
|         11|       25.0| 0.15555555555555556|
|         32|       27.5| 0.17777777777777778|
|         19|      29.99|                 0.2|
|         13|       34.0|  0.2222222222222222|
|         33|       34.0|  0.2222222222222222|
|         29|      35.98| 0.26666666666666666|
|         37|      39.98| 0.28888888888888886|
|          6|       40.0|  0.3111111111111111|
|         40|       40.0|  0.3111111111111111|
|         43|       40.0|  0.3111111111111111|
|         17|       40.0|  0.3111111111111111|
|         25|

Reflection Questions

1. Why do we convert continuous values into categories?

To simplify analysis, identify customer groups, and support business decisions like rewards and marketing.

2. What is the difference between business segmentation and technical bucketing?

Business segmentation: Uses business-defined rules (e.g., Gold if spending > ₹10,000).
Technical bucketing: Uses algorithms or statistical methods (e.g., quantiles or Bucketizer).

3. When would fixed thresholds fail?

When customer spending patterns change over time or differ significantly across datasets.

4. How does quantile-based segmentation differ from fixed rules?

Quantiles divide customers into groups with similar sizes based on the data distribution, while fixed rules use predefined spending limits.

5. Which method would you use in real-world projects?

It depends on the use case:

Fixed thresholds for stable business rules (loyalty programs).
Quantile-based segmentation when spending patterns change frequently.
Bucketizer for machine learning pipelines.